In [ ]:
DATASET_URL = "https://app.roboflow.com/ds/IZyryId9pu?key=ZrRsrwSRBZ"
MODEL_URL = "https://zenodo.org/records/10972956/files/CapricciosaX.pt?download=1"
MODEL_PATH = "CapricciosaX.pt"

#DATASET_URL = "https://app.roboflow.com/ds/oBG0mZue7H?key=BZXzboDm7N"
#MODEL_URL = ""
#MODEL_PATH = "1598FineTuned16150312_0101.pt"

DATASET_NAME = "paris-1667_X"
TRAIN_PROJECT_NAME = "runs_capricciosa"
TRAIN_NAME = f"{DATASET_NAME}_finetune"


In [ ]:
# # Clean install of torch and ultralytics to avoid torch._dynamo issues

# !pip uninstall -y torch torchvision torchaudio ultralytics

# # Install a stable GPU build of PyTorch (CUDA 12.1)
# !pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

# # Install ultralytics after torch
# !pip install -q ultralytics==8.3.0

!pip install -q ultralytics

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # hard disable Dynamo

import torch
from pathlib import Path
import zipfile
import shutil

from ultralytics import YOLO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.9.0+cu126
CUDA available: True


In [ ]:
from pathlib import Path
import zipfile
import shutil

# @title Download dataset
DATA_DIR = Path(DATASET_NAME)
ZIP_PATH = Path(f"{DATASET_NAME}.zip")

# Download only if missing
if not ZIP_PATH.exists():
    print("Dataset ZIP not found, downloading...")
    !wget -O {ZIP_PATH} "{DATASET_URL}"
else:
    print("Dataset ZIP already exists, skipping download.")

# Extract only if folder not present
if not DATA_DIR.exists():
    print(f"Extracting dataset {DATASET_NAME}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)
else:
    print("Dataset folder already exists, skipping extraction.")

print("Dataset contents:")
for p in DATA_DIR.rglob("*"):
    print(p)

NameError: name 'DATASET_NAME' is not defined

In [ ]:
DATA_YAML = str((DATA_DIR / "data.yaml").resolve())  # change if needed
print("Using data.yaml:", DATA_YAML)
print(open(DATA_YAML).read())

Using data.yaml: /content/paris-1667_X/data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 14
names: ['CatchWord', 'DigitizationArtefactZone', 'DropCapitalZone', 'DropCapitalZone-Plain', 'GraphicZone-Decoration', 'GraphicZone-Diagram', 'GraphicZone-Table', 'MainZone', 'MainZone-Head--Book', 'MainZone-Head--Section', 'MarginTextZone-RomanNumerals', 'NumberingZone', 'QuireMarksZone', 'RunningTitleZone']

roboflow:
  workspace: mia-workplace
  project: 1667-combined-gt
  version: 5
  license: CC BY 4.0
  url: https://universe.roboflow.com/mia-workplace/1667-combined-gt/dataset/5


In [ ]:
from pathlib import Path
import yaml

data_dir = Path(DATASET_NAME)
yaml_path = data_dir / "data.yaml"
print("yaml exists:", yaml_path.exists(), yaml_path)

cfg = yaml.safe_load(yaml_path.read_text())
print(cfg)

# resolve train/val folders and count images + labels
def count_pairs(split_path):
    p = (yaml_path.parent / split_path).resolve()
    imgs = list(p.rglob("*.jpg")) + list(p.rglob("*.png")) + list(p.rglob("*.jpeg"))
    return p, len(imgs)

for k in ["train", "val", "test"]:
    if k in cfg:
        p, n = count_pairs(cfg[k])
        print(k, "->", p, "images:", n)


yaml exists: True paris-1667_X/data.yaml
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 14, 'names': ['CatchWord', 'DigitizationArtefactZone', 'DropCapitalZone', 'DropCapitalZone-Plain', 'GraphicZone-Decoration', 'GraphicZone-Diagram', 'GraphicZone-Table', 'MainZone', 'MainZone-Head--Book', 'MainZone-Head--Section', 'MarginTextZone-RomanNumerals', 'NumberingZone', 'QuireMarksZone', 'RunningTitleZone'], 'roboflow': {'workspace': 'mia-workplace', 'project': '1667-combined-gt', 'version': 5, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/mia-workplace/1667-combined-gt/dataset/5'}}
train -> /content/train/images images: 0
val -> /content/valid/images images: 0
test -> /content/test/images images: 0


In [ ]:
# from pathlib import Path
# import yaml

# ds = Path("paris-1667")
# yaml_path = ds / "data.yaml"

# cfg = yaml.safe_load(yaml_path.read_text())

# # Roboflow sometimes uses "valid" instead of "val"
# cfg["path"] = str(ds.resolve())
# cfg["train"] = "train/images"
# cfg["val"] = "valid/images" if (ds / "valid/images").exists() else "val/images"
# cfg["test"] = "test/images" if (ds / "test/images").exists() else None

# # clean None keys
# cfg = {k: v for k, v in cfg.items() if v is not None}

# yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
# print("rewrote", yaml_path)
# print(yaml.safe_load(yaml_path.read_text()))


In [ ]:
# from pathlib import Path
# import yaml

# ds = Path("paris-1667")
# cfg = yaml.safe_load((ds/"data.yaml").read_text())
# base = Path(cfg["path"])

# for split in ["train", "val", "test"]:
#     if split in cfg:
#         p = (base / cfg[split]).resolve()
#         imgs = list(p.rglob("*.jpg")) + list(p.rglob("*.png")) + list(p.rglob("*.jpeg"))
#         print(split, "->", p, "images:", len(imgs))


In [ ]:
# from pathlib import Path

# # find some label files
# label_files = list(Path(DATASET_NAME).rglob("labels/*.txt"))
# print("num label files:", len(label_files))

# # sample a few lines
# for lf in label_files[:5]:
#     print(lf, "->", lf.read_text().splitlines()[:3])

# # check for out-of-range boxes
# bad = 0
# for lf in label_files:
#     for line in lf.read_text().splitlines():
#         parts = line.strip().split()
#         if len(parts) != 5:
#             bad += 1
#             continue
#         _, x, y, w, h = map(float, parts)
#         if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
#             bad += 1
#             break
# print("bad label files (format or range):", bad)


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# download only if missing
if not Path(MODEL_PATH).exists():
    print("Model not found, downloading...")
    !wget -O {Path(MODEL_PATH)} "{MODEL_URL}"
else:
    print("Model already exists, skipping download.")

# load custom checkpoint
model = YOLO(MODEL_PATH)

Model not found, downloading...
--2026-01-16 11:35:17--  https://zenodo.org/records/10972956/files/CapricciosaX.pt?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.185.43.153, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 136751294 (130M) [application/octet-stream]
Saving to: ‘CapricciosaX.pt’

CapricciosaX.pt     100%[===================>] 130.42M   600KB/s    in 3m 40s  

2026-01-16 11:38:59 (606 KB/s) - ‘CapricciosaX.pt’ saved [136751294/136751294]



Upgrading `torch` and `torchvision` to their latest versions to resolve potential compatibility issues. This might require a restart of the Colab runtime after execution.

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=50,         # tune this
    imgsz=640,
    batch=16,          # lower if you hit OOM
    workers=2,
    project=TRAIN_PROJECT_NAME,
    name=TRAIN_NAME,
    exist_ok=True,
)

# results = model.train(
#     data=DATA_YAML,
#     epochs=50,
#     imgsz=384,
#     cache=False,
#     batch=1,          # try 4, then 2, then 1
#     amp=True,         # mixed precision, usually big VRAM win
#     workers=2,
#     project=TRAIN_PROJECT_NAME,
#     name=TRAIN_NAME,
#     exist_ok=True,
# )

Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/paris-1667_X/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=CapricciosaX.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=paris-1667_X_finetune, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

In [ ]:
print(f"Download your model from {TRAIN_PROJECT_NAME}/{TRAIN_NAME}/weights/best.pt")